# 07 - Valores Ausentes

## Objetivo
Identificar, tratar e substituir valores ausentes (NaN).

## Conceitos

### O que sao valores ausentes
Pandas representa ausencia de dados com `NaN` (float) ou `None`.
Colunas com NaN passam a ter dtype float, o que pode surpreender.

### Deteccao
- `df.isna()`, `df.notna()`: mascara booleana.
- `df.isna().sum()`: contagem por coluna.
- `df.isna().mean()`: proporcao.
- `df.info()`: mostra contagem de nao nulos.

### Remocao
- `df.dropna()`: remove linhas com qualquer NaN.
- `df.dropna(axis=1)`: remove colunas.
- `df.dropna(subset=["col"])`: apenas nas colunas indicadas.
- `df.dropna(thresh=n)`: mantem linhas com pelo menos n valores nao nulos.

### Preenchimento
- `fillna(valor)`: preenche com constante.
- `fillna(df.mean())`: preenche com a media.
- `fillna(method="ffill")`: propaga ultimo valor valido.
- `fillna(method="bfill")`: propaga proximo valor valido.
- `interpolate()`: interpola linearmente.

### Estrategias
- Remover quando a ausencia e pequena e aleatoria.
- Preencher com media/mediana quando faz sentido.
- Criar indicador de ausencia para modelos.
- Investigar antes de decidir.

## DataFrame de exemplo com NaN
Criamos um DataFrame com valores ausentes em `idade`, `salario` e `cidade`
para praticar as tecnicas de deteccao, remocao e preenchimento.

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "nome": ["Ana", "Bruno", "Carla", "Diego", "Elisa", "Fabio"],
    "idade": [22, np.nan, 23, 28, np.nan, 35],
    "salario": [3500, 4200, np.nan, 5100, 3300, np.nan],
    "cidade": ["SP", "RJ", "MG", np.nan, "RJ", "MG"],
})
print("DataFrame:\n", df)

## Deteccao de NaN
- `isna().sum()`: contagem de NaN por coluna.
- `isna().mean()`: proporcao de NaN (util para decidir se vale remover).
- `info()`: mostra a contagem de valores nao nulos por coluna.
- `isna().any(axis=1)`: mascara de linhas com **algum** NaN.
- `notna().all(axis=1)`: mascara de linhas **sem** nenhum NaN.

In [ ]:
# Deteccao
print("\nNaN por coluna:\n", df.isna().sum())
print("\nProporcao de NaN:\n", df.isna().mean().round(2))
print("\ninfo:")
df.info()

# Linhas com algum NaN
print("\nLinhas com NaN:\n", df[df.isna().any(axis=1)])

# Linhas sem NaN
print("\nLinhas sem NaN:\n", df[df.notna().all(axis=1)])

## Remocao com dropna
- `dropna()`: remove qualquer linha que tenha **pelo menos um** NaN.
- `dropna(subset=["col"])`: so considera NaN na coluna indicada.
- `dropna(axis=1)`: remove **colunas** com NaN.
- `dropna(thresh=n)`: mantem linhas com pelo menos `n` valores nao nulos.

In [ ]:
# Removendo linhas com qualquer NaN
print("\ndropna():\n", df.dropna())

# Removendo apenas nas colunas indicadas
print("\ndropna subset:\n", df.dropna(subset=["idade"]))

# Removendo colunas com NaN
print("\ndropna axis=1:\n", df.dropna(axis=1))

# thresh: manter linhas com pelo menos 3 valores
print("\ndropna thresh=3:\n", df.dropna(thresh=3))

## Preenchimento com valor constante e estatisticas
- `fillna(valor)`: substitui NaN por uma constante.
- `fillna(media)` / `fillna(mediana)`: preenche com estatisticas da coluna.
- `fillna({col: valor, ...})`: valores diferentes por coluna.

In [ ]:
# Preenchendo com constante
print("\nfillna(0) na idade:\n", df["idade"].fillna(0))

# Preenchendo com a media
media_idade = df["idade"].mean()
print("\nMedia idade:", media_idade)
print("fillna media:\n", df["idade"].fillna(media_idade))

# Preenchendo com mediana
print("fillna mediana:\n", df["idade"].fillna(df["idade"].median()))

# Preenchendo colunas diferentes com valores diferentes
preenchido = df.fillna({"idade": df["idade"].mean(),
                        "salario": df["salario"].median(),
                        "cidade": "Desconhecida"})
print("\nPreenchimento por coluna:\n", preenchido)

## Propagacao (ffill / bfill)
- `ffill()` (forward fill): propaga o **ultimo** valor valido para baixo.
- `bfill()` (backward fill): propaga o **proximo** valor valido para cima.

Muito usados em series temporais, onde faz sentido assumir que o valor
se mantem entre medicoes.

In [ ]:
# Forward fill
print("\nffill:\n", df.ffill())

# Backward fill
print("\nbfill:\n", df.bfill())

## Interpolacao
`interpolate()` estima os valores ausentes a partir dos vizinhos,
assumindo uma progressao linear por padrao. Util em series numericas
com ausencias pontuais.

In [ ]:
# Interpolacao
s = pd.Series([1.0, np.nan, np.nan, 4.0, 5.0])
print("\nSerie com NaN:\n", s)
print("interpolate:\n", s.interpolate())

## Indicador de ausencia
Criar uma coluna binaria indicando se o valor era ausente pode ser
**informativo** em modelos — a ausencia, por si so, pode carregar sinal.

In [ ]:
# Indicador de ausencia
df_ind = df.copy()
df_ind["idade_ausente"] = df_ind["idade"].isna().astype(int)
print("\nCom indicador:\n", df_ind)